# Data Cleaning

In [1]:
import pandas as pd
import os

In [2]:
df = pd.read_csv("../data/raw/anime_data.csv")
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8852 entries, 0 to 8851
Data columns (total 20 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   mal_id        8852 non-null   int64  
 1   title         8852 non-null   str    
 2   source        8852 non-null   str    
 3   episodes      8513 non-null   float64
 4   synopsis      6492 non-null   str    
 5   year          6689 non-null   float64
 6   season        6689 non-null   str    
 7   producers     8852 non-null   str    
 8   genres        8852 non-null   str    
 9   studios       8852 non-null   str    
 10  demographics  8852 non-null   str    
 11  themes        8852 non-null   str    
 12  rating        8629 non-null   str    
 13  sequel        8852 non-null   bool   
 14  favorites     8852 non-null   int64  
 15  score         5538 non-null   float64
 16  wc            8852 non-null   int64  
 17  dropped       8852 non-null   int64  
 18  forum         8852 non-null   int64  
 

Let's remove duplicates.

In [4]:
df2 = df.drop_duplicates(subset=['mal_id'], keep='first')
df2.info()

<class 'pandas.DataFrame'>
Index: 8739 entries, 0 to 8851
Data columns (total 20 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   mal_id        8739 non-null   int64  
 1   title         8739 non-null   str    
 2   source        8739 non-null   str    
 3   episodes      8404 non-null   float64
 4   synopsis      6386 non-null   str    
 5   year          6581 non-null   float64
 6   season        6581 non-null   str    
 7   producers     8739 non-null   str    
 8   genres        8739 non-null   str    
 9   studios       8739 non-null   str    
 10  demographics  8739 non-null   str    
 11  themes        8739 non-null   str    
 12  rating        8517 non-null   str    
 13  sequel        8739 non-null   bool   
 14  favorites     8739 non-null   int64  
 15  score         5451 non-null   float64
 16  wc            8739 non-null   int64  
 17  dropped       8739 non-null   int64  
 18  forum         8739 non-null   int64  
 19  t

## Ongoing Anime

Tenrai API most likely has an "ongoing" variable, I just forgot to include it and I don't wanna redo 3 days of API calling. Luckily, if the episode count is null, then it's currently ongoing.

In [5]:
df3 = df2[~df2['episodes'].isna()]
df3.info()

<class 'pandas.DataFrame'>
Index: 8404 entries, 0 to 8845
Data columns (total 20 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   mal_id        8404 non-null   int64  
 1   title         8404 non-null   str    
 2   source        8404 non-null   str    
 3   episodes      8404 non-null   float64
 4   synopsis      6068 non-null   str    
 5   year          6405 non-null   float64
 6   season        6405 non-null   str    
 7   producers     8404 non-null   str    
 8   genres        8404 non-null   str    
 9   studios       8404 non-null   str    
 10  demographics  8404 non-null   str    
 11  themes        8404 non-null   str    
 12  rating        8337 non-null   str    
 13  sequel        8404 non-null   bool   
 14  favorites     8404 non-null   int64  
 15  score         5411 non-null   float64
 16  wc            8404 non-null   int64  
 17  dropped       8404 non-null   int64  
 18  forum         8404 non-null   int64  
 19  t

## Scoreless Anime

According to research, animes that have an N/A score are usually the obsucre and niche ones, or those super short ones that are only broadcast regionally. We will confirm this by looking at the wc and favorites count as well.

In [6]:
temp_df = df3[df3['score'].isna()]
temp_df['favorites'].describe()

count    2993.000000
mean        0.248580
std         4.684678
min         0.000000
25%         0.000000
50%         0.000000
75%         0.000000
max       254.000000
Name: favorites, dtype: float64

In [7]:
temp_df['wc'].describe()

count    2993.000000
mean       75.227531
std        65.759421
min         0.000000
25%        38.000000
50%        44.000000
75%        84.000000
max       606.000000
Name: wc, dtype: float64

Looking at the means, the scoreless anime really are irrelevant, and would just cloud statistical analysis and predictive modeling. Let's remove them.

In [8]:
df4 = df3[~df3['score'].isna()]
df4.info()

<class 'pandas.DataFrame'>
Index: 5411 entries, 0 to 8816
Data columns (total 20 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   mal_id        5411 non-null   int64  
 1   title         5411 non-null   str    
 2   source        5411 non-null   str    
 3   episodes      5411 non-null   float64
 4   synopsis      5372 non-null   str    
 5   year          5368 non-null   float64
 6   season        5368 non-null   str    
 7   producers     5411 non-null   str    
 8   genres        5411 non-null   str    
 9   studios       5411 non-null   str    
 10  demographics  5411 non-null   str    
 11  themes        5411 non-null   str    
 12  rating        5378 non-null   str    
 13  sequel        5411 non-null   bool   
 14  favorites     5411 non-null   int64  
 15  score         5411 non-null   float64
 16  wc            5411 non-null   int64  
 17  dropped       5411 non-null   int64  
 18  forum         5411 non-null   int64  
 19  t

Since I'm taking z-scores against cohort competitors (season & year), I would like to remove the few anime that do not have a year nor season.

In [9]:
df5 = df4.dropna(subset=['year', 'season'])
df5.info()

<class 'pandas.DataFrame'>
Index: 5368 entries, 0 to 8816
Data columns (total 20 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   mal_id        5368 non-null   int64  
 1   title         5368 non-null   str    
 2   source        5368 non-null   str    
 3   episodes      5368 non-null   float64
 4   synopsis      5331 non-null   str    
 5   year          5368 non-null   float64
 6   season        5368 non-null   str    
 7   producers     5368 non-null   str    
 8   genres        5368 non-null   str    
 9   studios       5368 non-null   str    
 10  demographics  5368 non-null   str    
 11  themes        5368 non-null   str    
 12  rating        5338 non-null   str    
 13  sequel        5368 non-null   bool   
 14  favorites     5368 non-null   int64  
 15  score         5368 non-null   float64
 16  wc            5368 non-null   int64  
 17  dropped       5368 non-null   int64  
 18  forum         5368 non-null   int64  
 19  t

No synopsis should be fine. My justification is that shows with no synopsis might be boring for users who look at MAL. However, categorizing by age rating is important: we need to know if restrictive shows have lower metrics.

In [10]:
df6 = df5[~df5['rating'].isna()]
df6.info()

<class 'pandas.DataFrame'>
Index: 5338 entries, 0 to 8816
Data columns (total 20 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   mal_id        5338 non-null   int64  
 1   title         5338 non-null   str    
 2   source        5338 non-null   str    
 3   episodes      5338 non-null   float64
 4   synopsis      5302 non-null   str    
 5   year          5338 non-null   float64
 6   season        5338 non-null   str    
 7   producers     5338 non-null   str    
 8   genres        5338 non-null   str    
 9   studios       5338 non-null   str    
 10  demographics  5338 non-null   str    
 11  themes        5338 non-null   str    
 12  rating        5338 non-null   str    
 13  sequel        5338 non-null   bool   
 14  favorites     5338 non-null   int64  
 15  score         5338 non-null   float64
 16  wc            5338 non-null   int64  
 17  dropped       5338 non-null   int64  
 18  forum         5338 non-null   int64  
 19  t

To enhance sentiment analysis, we can remove the endings of synopses that are like "(Written by...)" or "[Synopsis written by...]."

In [12]:
pattern = r"\s*[\(\[].*?[\)\]]\s*$"
df6["synopsis"] = df6["synopsis"].str.replace(pattern, "", regex=True)

For now, we can save this.

In [13]:
folder_path = "../data/processed"
file_name = "anime_data_1.csv"
full_path = os.path.join(folder_path, file_name)
df6.to_csv(full_path, index=False)
print(f"File successfully saved to: {full_path}")

File successfully saved to: ../data/processed\anime_data_1.csv
